# DB and Migrations Walkthrough

This notebook tours the persistence layer: migrations, connection pragmas,
schema readiness, and safety snapshots.


## Setup

Each walkthrough notebook uses its own scratch workspace under
`var/notebooks/` so you can rerun it without disturbing the others.


In [ ]:
from __future__ import annotations

import shutil
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src' / 'mtg_source_stack').exists():
            return candidate
    raise RuntimeError('Could not find the repo root from the current working directory.')


REPO_ROOT = find_repo_root(Path.cwd())
SRC_DIR = REPO_ROOT / 'src'
SRC_DIR_STR = str(SRC_DIR)
sys.path = [path for path in sys.path if path != SRC_DIR_STR]
sys.path.insert(0, SRC_DIR_STR)

WORK_DIR = REPO_ROOT / 'var' / 'notebooks' / '01_db_and_migrations'
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True)

from mtg_source_stack.db.connection import connect
from mtg_source_stack.db.migrator import current_schema_version, list_migration_files, migrate_database, pending_migrations
from mtg_source_stack.db.snapshots import create_database_snapshot, list_database_snapshots, restore_database_snapshot
from mtg_source_stack.inventory.report_helpers import render_table
from mtg_source_stack.inventory.service import create_inventory

DB_PATH = WORK_DIR / 'db_walkthrough.sqlite3'

print('repo root:', REPO_ROOT)
print('work dir:', WORK_DIR)
print('db path:', DB_PATH)


## Migration Files

The repo now uses ordered SQL migrations instead of relying on ordinary read
paths to upgrade the schema on the fly.


In [ ]:
migration_rows = [
    {
        'version': f'{migration.version:04d}',
        'name': migration.name,
        'resource': migration.resource_name,
    }
    for migration in list_migration_files()
]

print(render_table(migration_rows, [('version', 'version'), ('name', 'name'), ('resource', 'resource')]))


## Create and Stamp a Fresh Database

`migrate_database()` is the clearest way to show the current schema story on
a fresh file: apply every migration in order, then record what happened in
`schema_migrations`.


In [ ]:
applied = migrate_database(DB_PATH)

with connect(DB_PATH) as connection:
    version_rows = [dict(row) for row in connection.execute(
        'SELECT version, name, applied_at FROM schema_migrations ORDER BY version'
    )]
    table_rows = [dict(row) for row in connection.execute(
        """
        SELECT name, type
        FROM sqlite_master
        WHERE type = 'table' AND name NOT LIKE 'sqlite_%'
        ORDER BY name
        """
    )]

print('Applied migrations on this run:', [f'{row.version:04d} {row.name}' for row in applied])
print()
print('schema_migrations')
print(render_table(version_rows, [('version', 'version'), ('name', 'name'), ('applied_at', 'applied_at')]))
print()
print('Tables')
print(render_table(table_rows, [('name', 'name'), ('type', 'type')]))


## Connection Pragmas and Schema State

The connection helper enables foreign keys and also applies a few SQLite
pragmas that make the local-first database behave better when it is serving
an app, not just a one-shot CLI command.


In [ ]:
with connect(DB_PATH) as connection:
    pragma_rows = [
        {'pragma': 'foreign_keys', 'value': connection.execute('PRAGMA foreign_keys').fetchone()[0]},
        {'pragma': 'journal_mode', 'value': connection.execute('PRAGMA journal_mode').fetchone()[0]},
        {'pragma': 'busy_timeout', 'value': connection.execute('PRAGMA busy_timeout').fetchone()[0]},
        {'pragma': 'synchronous', 'value': connection.execute('PRAGMA synchronous').fetchone()[0]},
    ]
    schema_version = current_schema_version(connection)
    pending = pending_migrations(connection)

print(render_table(pragma_rows, [('pragma', 'pragma'), ('value', 'value')]))
print()
print('current schema version:', schema_version)
print('pending migrations:', [f'{migration.version:04d}' for migration in pending])


## Safety Snapshots and Restore

Snapshots are still the coarse backup layer. They are not the main audit
story anymore, but they remain useful when you want a quick point-in-time
recovery option for the whole database.


In [ ]:
empty_snapshot = create_database_snapshot(DB_PATH, label='fresh_schema')
create_inventory(
    DB_PATH,
    slug='personal',
    display_name='Personal Collection',
    description='Created after the empty snapshot',
)

with connect(DB_PATH) as connection:
    inventory_count_before_restore = connection.execute('SELECT COUNT(*) FROM inventories').fetchone()[0]

restore_result = restore_database_snapshot(DB_PATH, snapshot=empty_snapshot['snapshot_name'])

with connect(DB_PATH) as connection:
    inventory_count_after_restore = connection.execute('SELECT COUNT(*) FROM inventories').fetchone()[0]

snapshot_rows = [
    {
        'snapshot_name': row['snapshot_name'],
        'label': row['label'],
        'created_at': str(row['created_at']),
        'size_bytes': row['size_bytes'],
    }
    for row in list_database_snapshots(DB_PATH, limit=5)
]

print('inventory rows before restore:', inventory_count_before_restore)
print('inventory rows after restore:', inventory_count_after_restore)
print('restored snapshot:', restore_result['snapshot_name'])
if restore_result['pre_restore_snapshot'] is not None:
    print('pre-restore snapshot:', restore_result['pre_restore_snapshot']['snapshot_name'])
print()
print(render_table(snapshot_rows, [('snapshot_name', 'snapshot_name'), ('label', 'label'), ('created_at', 'created_at'), ('size_bytes', 'size_bytes')]))
